# Bead-pull calibration

Map the **stepper-motor step count** to the **bead position** for every pass of
the thread through the booster (each pass = one *sub-thread*).

For each sub-thread you record two motor positions:

* **start** &mdash; the bead-position zero of that sub-thread,
* **end** &mdash; the far end of the pass.

From those two numbers the code derives the travel *direction* and the scan
*length*, so nothing else has to be entered.  Positions are taken relative to
the homing limit switch (`gotoswitch`), so this calibration stays valid across
power cycles.

This notebook produces the **calibration file**: per sub-thread its `index`,
`name`, the non-measurement margins, the **(x, y, z) start/end points in the
booster frame**, and the measured `start_steps` / `end_steps`.  Those 3-D
endpoints let each measurement's 1-D position be mapped to a 3-D booster
coordinate.  The stepper settings come from the shared
`config/measurement_config.json`.  `steps_per_meter` (the step&harr;metre
conversion) can either be **read from that config** or **calibrated here**
(section 5) by jogging the bead across a known distance; a value calibrated here
is written straight back to the config so later runs use it.  Either way it is
only used for the printed summary and is **not** written to the calibration file.

**Workflow:** run the cells top-to-bottom.  Section 5 determines
`steps_per_meter`; section 6 is the loop you repeat for each sub-thread: jog the
bead to the start, capture it; jog to the end, capture it.

## 1. Imports and configuration

In [ ]:
import json
import sys
from pathlib import Path

# Make ``src`` importable (this notebook lives at the repo root).
sys.path.insert(0, str(Path.cwd()))

from src.bead_pull_controller import (
    Calibration,
    SubThreadCalibration,
    Stepper,
    SimulatedMotor,
)

# ---- edit these -----------------------------------------------------------
SIMULATE = True          # True: no hardware (dry test of the flow). False: real motor.

# How to obtain steps_per_meter (the step<->metre conversion):
#   "config"    : use the value stored in config/measurement_config.json.
#   "calibrate" : measure it in section 5 by jogging the bead across a known
#                 distance; the new value is written back to the config automatically.
STEPS_PER_METER_SOURCE = "calibrate"

# The passes of the thread through the booster you will calibrate, in order.
# For each pass set:
#   index          : integer id
#   name           : label (shown in logs)
#   margin_start_m : skip this many metres after the start (no measurement there)
#   margin_end_m   : skip this many metres before the end (no measurement there)
#   start_xyz_m    : (x, y, z) of the START point in the booster frame, metres
#   end_xyz_m      : (x, y, z) of the END point in the booster frame, metres
# The step anchors (start_steps / end_steps) are captured below by jogging.
SUB_THREADS = [
    {"index": 0, "name": "z0", "margin_start_m": 0.0, "margin_end_m": 0.0,
     "start_xyz_m": [-0.10, 0.0, 0.000], "end_xyz_m": [0.10, 0.0, 0.000]},
    {"index": 1, "name": "z1", "margin_start_m": 0.0, "margin_end_m": 0.0,
     "start_xyz_m": [-0.10, 0.0, 0.020], "end_xyz_m": [0.10, 0.0, 0.020]},
    {"index": 2, "name": "z2", "margin_start_m": 0.0, "margin_end_m": 0.0,
     "start_xyz_m": [-0.10, 0.0, 0.040], "end_xyz_m": [0.10, 0.0, 0.040]},
    {"index": 3, "name": "z3", "margin_start_m": 0.0, "margin_end_m": 0.0,
     "start_xyz_m": [-0.10, 0.0, 0.060], "end_xyz_m": [0.10, 0.0, 0.060]},
]

# ---- from the shared measurement config -----------------------------------
# Stepper settings live here.  _CONFIG_STEPS_PER_METER is the manual value stored
# in the config; STEPS_PER_METER starts from it and may be overwritten in
# section 5 if STEPS_PER_METER_SOURCE == "calibrate".
MEASUREMENT_CONFIG = Path("config/measurement_config.json")
_cfg = json.loads(MEASUREMENT_CONFIG.read_text(encoding="utf-8"))
STEPPER = _cfg.get("stepper", {})
_CONFIG_STEPS_PER_METER = _cfg.get("steps_per_meter", 1.0e5)
STEPS_PER_METER = _CONFIG_STEPS_PER_METER
CALIBRATION_PATH = Path(_cfg.get("calibration_file", "config/bead_pull_calibration.json"))

print(f"Stepper: port={STEPPER.get('port')}, baud={STEPPER.get('baud')}, motor={STEPPER.get('motor')}")
print(f"steps_per_meter source={STEPS_PER_METER_SOURCE!r}; config value={_CONFIG_STEPS_PER_METER:.6g}")
print(f"sub-threads to calibrate: {[d['index'] for d in SUB_THREADS]}")
print(f"Calibration will be saved to: {CALIBRATION_PATH}")

## 2. Connect to the motor

In [ ]:
if SIMULATE:
    motor = SimulatedMotor(verbose=True)
    print("SIMULATE = True: using an in-memory motor (no serial port opened).")
else:
    motor = Stepper.open(**STEPPER)   # the config keys match Stepper.open's arguments
    print("Connected to", STEPPER.get("port"))

## 3. Home to the limit switch

This drives the motor to the limit switch and zeroes the step counter, so every
position you capture below is measured from the same physical reference.

In [ ]:
motor.home()
print("Homed. Position now:", motor.get_position(), "steps")


## 4. Jog helpers

`jog(steps)` moves the bead by a relative number of steps (positive vs. negative
= the two directions; start with small values and watch which way the bead goes)
and prints the new absolute position.  `pos()` just prints the current position.
These helpers are used both to calibrate `steps_per_meter` (section 5) and to
capture each sub-thread's endpoints (section 6).

In [ ]:
def jog(steps):
    """Move the bead by a relative number of steps and report the new position."""
    motor.move_by(int(steps))
    p = motor.get_position()
    print(f"jogged {int(steps):+d} -> {p} steps")
    return p

def pos():
    p = motor.get_position()
    print("position:", p, "steps")
    return p

print("Helpers ready: jog(steps), pos().")

### Jog cell — run/edit this as many times as you need

In [ ]:
# Examples (edit the number, re-run):
jog(500)
# jog(-200)
# pos()


## 5. Determine `steps_per_meter`

`steps_per_meter` is the step&harr;metre conversion (steps = metres &times;
`steps_per_meter`).  Pick how to obtain it with `STEPS_PER_METER_SOURCE` in
section 1:

* **`"config"`** &mdash; use the value already stored in
  `config/measurement_config.json` (nothing to jog; just run the compute cell).
* **`"calibrate"`** &mdash; measure it now.  Set a known travel `CALIB_DISTANCE_M`,
  jog the bead to the **start** of that distance and `capture_spm('start')`, then
  jog it to the **end** and `capture_spm('end')`.  The compute cell sets
  `steps_per_meter = |end_steps - start_steps| / CALIB_DISTANCE_M` **and writes it
  straight back to `config/measurement_config.json`** so later measurement runs
  use the recalibrated value.

The jog helpers (`jog`, `pos`) from section 4 are reused here.

> **Dry test (`SIMULATE = True`):** there is no real bead, so just move the
> simulated motor between captures, e.g. `motor.move_by(0); capture_spm('start')`
> then `motor.move_by(20000); capture_spm('end')` with `CALIB_DISTANCE_M = 0.20`
> to get `steps_per_meter = 100000` (which is then written to the config).

In [ ]:
# Distance you will jog the bead across for the steps_per_meter calibration.
# Only used when STEPS_PER_METER_SOURCE == "calibrate".
CALIB_DISTANCE_M = 0.20        # metres between the two marks you will jog to

_spm = {}   # "start"/"end" -> motor step count

def capture_spm(which):
    """Record the current motor position as the start/end of the steps_per_meter
    calibration distance (uses the same motor and jog() as the sub-thread capture)."""
    which = which.lower()
    assert which in ("start", "end"), "which must be 'start' or 'end'"
    p = motor.get_position()
    _spm[which] = p
    print(f"steps_per_meter {which} = {p} steps")
    return p

print(f"STEPS_PER_METER_SOURCE = {STEPS_PER_METER_SOURCE!r}")
if STEPS_PER_METER_SOURCE == "calibrate":
    print(f"Jog across CALIB_DISTANCE_M = {CALIB_DISTANCE_M} m, capturing start then end.")
    print("Use jog(...) to move, then capture_spm('start') / capture_spm('end').")
else:
    print("Using the config value; skip the jog/capture cells and run the compute cell.")

In [ ]:
# CALIBRATE step: jog the bead to the START of CALIB_DISTANCE_M, then capture.
# jog(2000)   # <- move until the bead sits at the start mark
capture_spm("start")

In [ ]:
# CALIBRATE step: jog the bead to the END (CALIB_DISTANCE_M away), then capture.
# jog(20000)  # <- move until the bead sits at the end mark
capture_spm("end")

In [ ]:
# Set STEPS_PER_METER from the chosen source.  When calibrated, the new value is
# written straight back to config/measurement_config.json (no flag) so later
# measurement runs use it; the "config" source leaves the file untouched.
if STEPS_PER_METER_SOURCE == "calibrate":
    if "start" not in _spm or "end" not in _spm:
        raise ValueError("Capture both start and end (capture_spm) before computing.")
    if CALIB_DISTANCE_M <= 0:
        raise ValueError("CALIB_DISTANCE_M must be positive.")
    steps_travelled = abs(_spm["end"] - _spm["start"])
    if steps_travelled == 0:
        raise ValueError("start and end are the same position; re-jog and re-capture.")
    STEPS_PER_METER = steps_travelled / CALIB_DISTANCE_M
    print(f"Travelled {steps_travelled} steps over {CALIB_DISTANCE_M} m")
    print(f"Calibrated steps_per_meter = {STEPS_PER_METER:.6g} "
          f"(config value was {_CONFIG_STEPS_PER_METER:.6g})")
    # recalibration done -> persist to the shared config
    _cfg_data = json.loads(MEASUREMENT_CONFIG.read_text(encoding="utf-8"))
    _cfg_data["steps_per_meter"] = float(STEPS_PER_METER)
    MEASUREMENT_CONFIG.write_text(json.dumps(_cfg_data, indent=2) + "\n", encoding="utf-8")
    print(f"Wrote steps_per_meter to {MEASUREMENT_CONFIG}")
elif STEPS_PER_METER_SOURCE == "config":
    STEPS_PER_METER = _CONFIG_STEPS_PER_METER
    print(f"Using steps_per_meter from config: {STEPS_PER_METER:.6g}")
else:
    raise ValueError(
        f"STEPS_PER_METER_SOURCE must be 'config' or 'calibrate', got {STEPS_PER_METER_SOURCE!r}"
    )

## 6. Capture each sub-thread's start / end

Using `jog(...)` / `pos()` from section 4, for **each** sub-thread:

1. `jog(...)` until the bead sits at the sub-thread's **start** (zero), then run
   `capture(index, "start")`.
2. `jog(...)` until the bead sits at the sub-thread's **end**, then run
   `capture(index, "end")`.

You can revisit any sub-thread and capture again to overwrite a bad value.

In [ ]:
captured = {}   # index -> {"start_steps": ..., "end_steps": ...}

def capture(index, which):
    """Record the current position as the start/end of a sub-thread."""
    which = which.lower()
    assert which in ("start", "end"), "which must be 'start' or 'end'"
    p = motor.get_position()
    captured.setdefault(index, {})[f"{which}_steps"] = p
    print(f"sub-thread {index}: {which} = {p} steps")
    return p

print("Helper ready: capture(index, 'start'|'end').  (jog/pos come from section 4.)")

### Capture cells — run when the bead is at the start / end of a sub-thread

In [ ]:
# When the bead is at the START (zero) of, say, sub-thread 0:
capture(0, "start")


In [ ]:
# When the bead is at the END of sub-thread 0:
capture(0, "end")


Repeat the jog + capture cells for every entry in `SUB_THREADS` (indices
`1`, `2`, ... here).  You can copy the two capture lines and change the index, or
just re-run them after editing the index.  Check progress any time:

In [ ]:
for d in SUB_THREADS:
    i = d["index"]
    got = captured.get(i, {})
    print(f"sub-thread {i} ({d['name']}): "
          f"start={got.get('start_steps', '---')}, end={got.get('end_steps', '---')}")


> **Tip for a dry test (`SIMULATE = True`):** there is no real bead, so just move
> the simulated motor before capturing, e.g. `motor.move_by(0); capture(0,'start')`
> then `motor.move_by(20000); capture(0,'end')`, to exercise the save below.

## 7. Build and save the calibration

In [ ]:
sub_threads = []
for d in SUB_THREADS:
    i = d["index"]
    if i not in captured or "start_steps" not in captured[i] or "end_steps" not in captured[i]:
        raise ValueError(f"sub-thread {i} ({d.get('name')}) is missing a start or end capture.")
    sub_threads.append(
        SubThreadCalibration(
            index=i,
            name=d.get("name"),
            margin_start_m=d.get("margin_start_m", 0.0),
            margin_end_m=d.get("margin_end_m", 0.0),
            start_xyz_m=d.get("start_xyz_m"),
            end_xyz_m=d.get("end_xyz_m"),
            start_steps=captured[i]["start_steps"],
            end_steps=captured[i]["end_steps"],
        )
    )

# steps_per_meter (from config or calibrated in section 5) is NOT written to the
# calibration file; it is only used here for the printed summary (length_m column).
calibration = Calibration(steps_per_meter=STEPS_PER_METER, sub_threads=sub_threads)
print(calibration.summary())
path = calibration.save(CALIBRATION_PATH)
print("\nSaved calibration to:", path)

## 8. Verify (reload from disk)

In [ ]:
# Reload the saved calibration (anchors + name + 3-D endpoints + margins) and
# attach the steps_per_meter determined above (from config or calibrated),
# exactly as run_bead_pull_measurement.py does at load time.
reloaded = Calibration.from_calibration_file(CALIBRATION_PATH, STEPS_PER_METER)
print(reloaded.summary())

## 9. Release the motor

Switches the coils off and closes the serial port (no-op for the simulated
motor).

In [ ]:
motor.shutdown()
print("Motor released.")
